In [6]:
import numpy as np    #np, mpl and pd serve as a nicknames
import matplotlib as mpl
import matplotlib.pyplot as plt
import pandas as pd
import os
import dss #OpenDss Module
    

In [7]:
os.chdir('C:\\Users\\amy\\projects\\opendss\\trab\\')   # force going to the working folder (ADJUST FOR YOUR OWN PATH!)
mydir=os.getcwd()                          # mydir stores the working path
print(mydir) 

C:\Users\amy\projects\opendss\trab


In [8]:
dss_engine = dss.DSS
DSSText = dss_engine.Text                                                      
DSSCircuit = dss_engine.ActiveCircuit                                            
DSSSolution = dss_engine.ActiveCircuit.Solution                                      
ControlQueue = dss_engine.ActiveCircuit.CtrlQueue   

In [9]:
DSSText.Command = 'Compile ' + mydir + '\\ieee37.dss'

In [10]:
DSSCircuit = dss_engine.ActiveCircuit                                            
DSSCircuit.AllBusNames

['sourcebus',
 '799',
 '709',
 '775',
 '701',
 '702',
 '705',
 '713',
 '703',
 '727',
 '730',
 '704',
 '714',
 '720',
 '742',
 '712',
 '706',
 '725',
 '707',
 '724',
 '722',
 '708',
 '733',
 '732',
 '731',
 '710',
 '735',
 '736',
 '711',
 '741',
 '740',
 '718',
 '744',
 '734',
 '737',
 '738',
 '728',
 '729',
 '799r']

In [11]:
# Get all element names and store in variant array of strings
Elenames = DSSCircuit.AllElementNames
print('List of elements')
print ('Total:' +  str(len(Elenames)))
print(Elenames)

# Get all load names and store in variant array of strings
Loadnames = DSSCircuit.Loads.AllNames
print('List of Loads')
print ('Total:' +  str(len(Loadnames)))
print(Loadnames)

# Get all bus names and store in variant array of strings
Busnames = DSSCircuit.AllBusNames
print('List of Buses')
print ('Total:' +  str(len(Busnames)))
print(Busnames)

# Get all transformers names and store in variant array of strings
Trafonames = DSSCircuit.Transformers.AllNames
print('List of Transformers')
print ('Total:' +  str(len(Trafonames)))
print(Trafonames)

# Get all line names and store in variant array of strings
Linenames = DSSCircuit.Lines.AllNames
print('List of Lines')
print ('Total:' +  str(len(Linenames)))
print(Linenames)

List of elements
Total:73
['Vsource.source', 'Transformer.subxf', 'Transformer.xfm1', 'Line.l1', 'Line.l2', 'Line.l3', 'Line.l4', 'Line.l5', 'Line.l6', 'Line.l7', 'Line.l8', 'Line.l9', 'Line.l10', 'Line.l11', 'Line.l12', 'Line.l13', 'Line.l14', 'Line.l15', 'Line.l16', 'Line.l17', 'Line.l18', 'Line.l19', 'Line.l20', 'Line.l21', 'Line.l22', 'Line.l23', 'Line.l24', 'Line.l25', 'Line.l26', 'Line.l27', 'Line.l28', 'Line.l29', 'Line.l30', 'Line.l31', 'Line.l32', 'Line.l33', 'Line.l34', 'Line.l35', 'Transformer.reg1a', 'RegControl.creg1a', 'Transformer.reg1c', 'RegControl.creg1c', 'Line.jumper', 'Load.s701a', 'Load.s701b', 'Load.s701c', 'Load.s712c', 'Load.s713c', 'Load.s714a', 'Load.s714b', 'Load.s718a', 'Load.s720c', 'Load.s722b', 'Load.s722c', 'Load.s724b', 'Load.s725b', 'Load.s727c', 'Load.s728', 'Load.s729a', 'Load.s730c', 'Load.s731b', 'Load.s732c', 'Load.s733a', 'Load.s734c', 'Load.s735c', 'Load.s736b', 'Load.s737a', 'Load.s738a', 'Load.s740c', 'Load.s741c', 'Load.s742a', 'Load.s742b',

In [12]:
DSSText.Command = 'set loadmult = 0.2'

In [13]:
#Configure power flow options
DSSText.Command = 'reset'
DSSText.Command = 'set controlmode=static'
DSSText.Command = 'set mode=snapshot' 

DSSSolution.Solve()#Solve power flow

In [14]:
#Power and Losses
kW_BC = DSSCircuit.TotalPower[0] #in kW
kvar_BC = DSSCircuit.TotalPower[1] #in kVAr
kW_Loss_BC = DSSCircuit.Losses[0]/1000 #property returns values in W, here stored in kW
kvar_Loss_BC = DSSCircuit.Losses[1]/1000 #property returns values in VAr, here stored in kVAr
print("Total kW delivered to the network: %.3f kW" %kW_BC)
print("Total kVAr delivered to the network: %.3f kvar" %kvar_BC)
print("Total active losses: %.3f kW" %kW_Loss_BC)
print("Percent losses: %.3f" %abs(((kW_Loss_BC/kW_BC)*100)) + "%")
print("Total reactive losses: %.3f kVAr" %kvar_Loss_BC)

Total kW delivered to the network: -502.732 kW
Total kVAr delivered to the network: -250.824 kvar
Total active losses: 5.475 kW
Percent losses: 1.089%
Total reactive losses: 6.772 kVAr


In [15]:
#Voltage
#DSSCircuit.AllNodeVmagPUByPhase()  
#DSSCircuit.AllNodeNamesByPhase)
nodesA_filtered = []
busVpuA_BC_filtered = []
nodesB_filtered = []
busVpuB_BC_filtered = []
nodesC_filtered = []
busVpuC_BC_filtered = []

busVpuA_BC=(DSSCircuit.AllNodeVmagPUByPhase(1))#phase 1
nodesA=(DSSCircuit.AllNodeNamesByPhase(1))#phase 1
for node, voltage in zip(nodesA, busVpuA_BC):#remove voltage regulators
    if not node.startswith("reg"):
        nodesA_filtered.append(node)
        busVpuA_BC_filtered.append(voltage)
        
busVpuB_BC=(DSSCircuit.AllNodeVmagPUByPhase(2))#phase 2
nodesB=(DSSCircuit.AllNodeNamesByPhase(2))#phase 2
for node, voltage in zip(nodesB, busVpuB_BC):#remove voltage regulators
    if not node.startswith("reg"):
        nodesB_filtered.append(node)
        busVpuB_BC_filtered.append(voltage)
        
busVpuC_BC=(DSSCircuit.AllNodeVmagPUByPhase(3))#phase 3
nodesC=(DSSCircuit.AllNodeNamesByPhase(3))#phase 3
for node, voltage in zip(nodesC, busVpuC_BC):#remove voltage regulators
    if not node.startswith("reg"):
        nodesC_filtered.append(node)
        busVpuC_BC_filtered.append(voltage)



In [16]:
# Transformers AMPS loading 
#This function gets transformer current data:all phases current magnitude at HV(wdg1) side winding,
# all phases Amps loading at HV side 

#List of transformers (excluding the voltage regulators)
onlytrafo = [trafo for trafo in Trafonames if not trafo.startswith('reg')]

#Initialize variables
All_Wdg_Amp = np.zeros((len(onlytrafo),24))#to store all phases winding currents at HV and LV sides in complex form (24 entries for each three phase transformer)
Transf_Wdg1_Amps = np.zeros((len(onlytrafo),3))#to store all phases current magnitude at HV side winding )   
Transf_Amps_loading = np.zeros((len(onlytrafo),3))#to store all phases Amps loading at HV side 
Transf_Max_Amps_loading = np.zeros((len(onlytrafo)))#to store maximun amps loading at HV side (the maximun of the three phases)       
#------------------------------
for itrafo, trafo in enumerate(onlytrafo):
          DSSCircuit.SetActiveElement('Transformer.%s' %(trafo))
          DSSCircuit.Transformers.Name = trafo
          DSSCircuit.Transformers.Wdg = 1
          amp_rate = DSSCircuit.ActiveElement.NormalAmps#get Normal ampere rating (OpenDSS defines it for Winding 1)
          All_Wdg_Amp[itrafo,:] = DSSCircuit.Transformers.WdgCurrents#get all winding currents at HV and LV sides in complex form  
          Transf_Wdg1_Amps[itrafo,0]=(np.sqrt(All_Wdg_Amp[itrafo,0]**2+All_Wdg_Amp[itrafo,1]**2))#HV side Wdg current , phase 1
          Transf_Wdg1_Amps[itrafo,1]=(np.sqrt(All_Wdg_Amp[itrafo,8]**2+All_Wdg_Amp[itrafo,9]**2)) #HV side Wdg current , phase 2
          Transf_Wdg1_Amps[itrafo,2]=(np.sqrt(All_Wdg_Amp[itrafo,16]**2+All_Wdg_Amp[itrafo,17]**2))#HV side Wdg current , phase 3
    
          Transf_Amps_loading[itrafo,0]=Transf_Wdg1_Amps[itrafo,0]/amp_rate #HV side Wdg current loading, phase 1 
          Transf_Amps_loading[itrafo,1]=Transf_Wdg1_Amps[itrafo,1]/amp_rate #HV side Wdg current loading, phase 2
          Transf_Amps_loading[itrafo,2]=Transf_Wdg1_Amps[itrafo,2]/amp_rate #HV side Wdg current loading, phase 3

In [17]:
display(Transf_Wdg1_Amps)
display(All_Wdg_Amp)

array([[7.82909419e-01, 7.31874982e-01, 9.45553529e-01],
       [2.12667839e-05, 2.13692101e-05, 2.12327130e-05]])

array([[ 7.78822706e-01, -7.98896158e-02, -7.78823612e-01,
         7.98890928e-02, -3.73185904e+01,  3.82794573e+00,
         3.73185437e+01, -3.82796757e+00, -2.34513733e-01,
        -6.93285150e-01,  2.34513733e-01,  6.93286196e-01,
         1.12370286e+01,  3.32199649e+01, -1.12370312e+01,
        -3.32199133e+01, -5.44308974e-01,  7.73174765e-01,
         5.44309880e-01, -7.73175288e-01,  2.60815568e+01,
        -3.70479073e+01, -2.60815174e+01,  3.70478842e+01],
       [ 5.64547247e-06, -2.05037737e-05, -1.45964249e-05,
         1.55706183e-05, -5.92038396e-05,  1.09705616e-06,
        -3.02868466e-05, -5.04377913e-05, -2.06537936e-05,
         5.48306014e-06,  2.07888945e-05,  4.71512135e-06,
         3.02868430e-05,  5.04377876e-05, -2.89169966e-05,
         5.15348438e-05,  1.50054955e-05,  1.50220906e-05,
        -6.19529571e-06, -2.02843626e-05,  2.89169944e-05,
        -5.15348420e-05,  5.92038415e-05, -1.09705616e-06]])

In [18]:
#**************--------------Lines Amps Loading
Line_Max_Amp = np.zeros((len(Linenames)))
for iline, line in enumerate(Linenames):
        DSSCircuit.SetActiveElement('line.%s' %(line))
        amp_rate = DSSCircuit.ActiveElement.NormalAmps
        Line_Max_Amp[iline] = max(DSSCircuit.ActiveElement.CurrentsMagAng[0:12:2])/amp_rate


In [19]:
#Just to check what the function returns
#DSSCircuit.SetActiveElement('line.650632')
#max(DSSCircuit.ActiveElement.CurrentsMagAng[0:12:2])
#DSSCircuit.ActiveElement.NormalAmps